# Ekstraksi Data Kualitas Udara - Kabupaten Bangkalan

Notebook ini menarik data satelit **Sentinel-5P (TROPOMI) Level 2** dari
**Copernicus Data Space Ecosystem** melalui antarmuka **openEO**, khusus untuk
wilayah **Kabupaten Bangkalan, Jawa Timur**.

**Polutan yang diambil:** NO2, CO, SO2, CH4

**Rentang waktu:** 24 Agustus 2025 sampai dengan 24 Agustus 2026

**Area of Interest (AOI):** batas administratif asli Kabupaten Bangkalan
(bentuk polygon sesuai wilayah sebenarnya, bukan kotak bounding box).

**Alur kerja notebook ini:**
1. Import pustaka yang dibutuhkan
2. Konfigurasi AOI (polygon asli Bangkalan), rentang waktu, dan folder output
3. Autentikasi ke backend openEO Copernicus Data Space (device code flow)
4. Membuat fungsi ekstraksi generik untuk satu polutan
5. Uji coba tunggal dan diagnostik, sebelum menjalankan ekstraksi penuh
   setahun untuk keempat polutan
6. Menjalankan ekstraksi untuk keempat polutan dan menyimpan hasilnya
   sebagai file NetCDF (`.nc`)
7. Menggabungkan hasil keempat polutan menjadi **satu tabel CSV**, mirip
   struktur tabel pada basis data, yang siap dipakai pada notebook analisis

> **Catatan penting:** Backend openEO Sentinel Hub untuk koleksi
> `SENTINEL_5P_L2` hanya mengizinkan **satu band/polutan per request**.
> Karena itu, proses ekstraksi dilakukan dengan cara loop satu per satu
> untuk setiap polutan, bukan sekaligus dalam satu `load_collection`.


## 1. Import Pustaka

- `openeo`, klien Python resmi untuk berkomunikasi dengan backend openEO.
- `os`, membuat folder output jika belum ada.
- `json`, membaca file GeoJSON batas wilayah Bangkalan.
- `xarray`, membaca file NetCDF hasil ekstraksi.
- `pandas`, mengonversi dan menggabungkan data menjadi satu tabel.
- `time`, jeda kecil antar request agar tidak membanjiri backend.


In [3]:
import json
import os
import time

import openeo
import numpy as np
import pandas as pd
import xarray as xr


## 2. Konfigurasi Umum

Bagian ini berisi semua parameter yang mungkin perlu diubah:
- **URL backend** openEO Copernicus Data Space.
- **Polygon AOI**, batas administratif asli Kabupaten Bangkalan (bukan kotak).
- **Rentang waktu** pengambilan data.
- **Daftar polutan** yang akan diekstraksi beserta nama band-nya di koleksi
  `SENTINEL_5P_L2`.
- **Folder output** untuk file NetCDF dan CSV.

Polygon di bawah diambil dari data batas administratif resmi (GADM level 2,
kode wilayah 3526), sudah disederhanakan agar tidak terlalu berat saat
dikirim sebagai parameter permintaan ke openEO, namun tetap mengikuti bentuk
asli Kabupaten Bangkalan, termasuk lekukan pesisir dan bukan sekadar kotak
persegi.


In [ ]:
# --- Backend openEO Copernicus Data Space ---
BACKEND_URL = "openeo.dataspace.copernicus.eu"

# Koordinat batas administratif Kabupaten Bangkalan (bentuk asli, bukan kotak),
# disederhanakan dari data GADM level-2 (sumber: mahendrayudha/indonesia-geojson,
# kode wilayah BPS/Kemendagri 3526). Urutan tiap titik adalah [longitude, latitude].
BANGKALAN_BOUNDARY_COORDS = [
    [112.992577, -7.202438], [112.964821, -7.199743], [112.952950, -7.192856], [112.891159, -7.178034],
    [112.864861, -7.162668], [112.845116, -7.165686], [112.800705, -7.157525], [112.790970, -7.161581],
    [112.776932, -7.159422], [112.769234, -7.162904], [112.765221, -7.152152], [112.749481, -7.150523],
    [112.751534, -7.153605], [112.740387, -7.165307], [112.744507, -7.168972], [112.746956, -7.167730],
    [112.745903, -7.170421], [112.729523, -7.170129], [112.722351, -7.175711], [112.705925, -7.167363],
    [112.691147, -7.153987], [112.703835, -7.113265], [112.705864, -7.096297], [112.701637, -7.092802],
    [112.691666, -7.096992], [112.685768, -7.095528], [112.686516, -7.089311], [112.673424, -7.073466],
    [112.673592, -7.059717], [112.676514, -7.056869], [112.680687, -7.034704], [112.688461, -7.029723],
    [112.711067, -7.040588], [112.723526, -7.038418], [112.740555, -7.018154], [112.770447, -7.000629],
    [112.788673, -6.984257], [112.792824, -6.978434], [112.790848, -6.973830], [112.804016, -6.963901],
    [112.828331, -6.930720], [112.826485, -6.927431], [112.829796, -6.921215], [112.822350, -6.912736],
    [112.823563, -6.910023], [112.832245, -6.906312], [112.835457, -6.915974], [112.841042, -6.916435],
    [112.848381, -6.908895], [112.847359, -6.898939], [112.859215, -6.893069], [112.871254, -6.891851],
    [112.869896, -6.893898], [112.875259, -6.897953], [112.879013, -6.896546], [112.877518, -6.891464],
    [112.882042, -6.890491], [112.920799, -6.893889], [112.925453, -6.889861], [112.933083, -6.891401],
    [112.951904, -6.886978], [112.961586, -6.889064], [112.963661, -6.886701], [112.970543, -6.888798],
    [112.978485, -6.885591], [112.987343, -6.886865], [112.992882, -6.883408], [112.995811, -6.885634],
    [112.999809, -6.881589], [113.002197, -6.887471], [113.006149, -6.881138], [113.025635, -6.881006],
    [113.023659, -6.882414], [113.027351, -6.883591], [113.033890, -6.879476], [113.052658, -6.884909],
    [113.063591, -6.882000], [113.077377, -6.883579], [113.087448, -6.888769], [113.107979, -6.888528],
    [113.110619, -6.894425], [113.117241, -6.894449], [113.121170, -6.889987], [113.127182, -6.898141],
    [113.127533, -6.904702], [113.118767, -6.905104], [113.115997, -6.935714], [113.111214, -6.937359],
    [113.101646, -6.931810], [113.095100, -6.945797], [113.099190, -6.951012], [113.108170, -6.953403],
    [113.102470, -6.959991], [113.106155, -6.972141], [113.109398, -6.974140], [113.103928, -6.985951],
    [113.112564, -6.988916], [113.133842, -6.976145], [113.147812, -6.984626], [113.144753, -6.989298],
    [113.146271, -6.997767], [113.140907, -7.000000], [113.138756, -7.005353], [113.142281, -7.014079],
    [113.133591, -7.023641], [113.133476, -7.035540], [113.127853, -7.036573], [113.115227, -7.031735],
    [113.116089, -7.049525], [113.110649, -7.047163], [113.113457, -7.053055], [113.111969, -7.059690],
    [113.119843, -7.066993], [113.120209, -7.079932], [113.116776, -7.082044], [113.114525, -7.078621],
    [113.108124, -7.077841], [113.111946, -7.095151], [113.116127, -7.098413], [113.115921, -7.104572],
    [113.112671, -7.105823], [113.115540, -7.116165], [113.107277, -7.107961], [113.107567, -7.117204],
    [113.103828, -7.125473], [113.096153, -7.127437], [113.092163, -7.133172], [113.095032, -7.141696],
    [113.093582, -7.151216], [113.097397, -7.156816], [113.091858, -7.157668], [113.088264, -7.156389],
    [113.088081, -7.152215], [113.077011, -7.151009], [113.070190, -7.162508], [113.057594, -7.163670],
    [113.054810, -7.169858], [113.056465, -7.174946], [113.049835, -7.178644], [113.048210, -7.184325],
    [113.061234, -7.190079], [113.060951, -7.196105], [113.050308, -7.195768], [113.040367, -7.212882],
    [113.013832, -7.205460], [113.000648, -7.209431], [112.992577, -7.202438],
]

bangkalan_polygon = {
    "type": "Polygon",
    "coordinates": [BANGKALAN_BOUNDARY_COORDS],
}

# Bounding box diturunkan otomatis dari polygon di atas, hanya dipakai sebagai
# batas kasar (spatial_extent) saat load_collection. Agregasi spasial yang
# sesungguhnya tetap memakai bentuk polygon asli, bukan kotak ini.
_lons = [pt[0] for pt in BANGKALAN_BOUNDARY_COORDS]
_lats = [pt[1] for pt in BANGKALAN_BOUNDARY_COORDS]
bangkalan_bbox = {
    "west": min(_lons),
    "east": max(_lons),
    "south": min(_lats),
    "north": max(_lats),
}

# --- Rentang waktu ekstraksi ---
START_DATE = "2025-08-24"
END_DATE = "2026-08-24"

# --- Daftar polutan yang akan diekstraksi ---
# key   = nama pendek yang dipakai untuk penamaan file dan kolom tabel
# value = nama band pada koleksi SENTINEL_5P_L2
POLLUTANTS = {
    "NO2": "NO2",
    "CO": "CO",
    "SO2": "SO2",
    "CH4": "CH4",
}

# --- Folder output ---
NC_DIR = "../data/nc/"

# Tabel gabungan hasil ekstraksi disimpan satu folder dengan notebook ini
# (folder "materi/"), mengikuti struktur proyek yang sudah dipakai.
COMBINED_CSV_PATH = "data_polutan_bangkalan.csv"

os.makedirs(NC_DIR, exist_ok=True)

print("Konfigurasi siap.")
print("Bounding box (kasar, dari polygon):", bangkalan_bbox)
print("Jumlah titik polygon AOI:", len(bangkalan_polygon["coordinates"][0]))
print("Polutan yang akan diekstraksi:", list(POLLUTANTS.keys()))


## 3. Autentikasi ke Copernicus Data Space (Device Code Flow)

`connection.authenticate_oidc()` akan mencoba beberapa metode autentikasi
OpenID Connect secara berurutan, dan jika tidak ada sesi tersimpan, ia akan
otomatis jatuh ke device code flow, klien akan mencetak sebuah tautan dan
kode singkat di terminal atau output sel.

**Langkah yang perlu kamu lakukan saat sel ini dijalankan:**
1. Buka tautan yang muncul di output (`https://.../device`).
2. Login dengan akun Copernicus Data Space kamu.
3. Masukkan kode yang ditampilkan di output notebook.
4. Tunggu hingga sel selesai, jika berhasil akan muncul pesan konfirmasi
   otentikasi.

Setelah berhasil sekali, token biasanya akan disimpan secara lokal (cache)
sehingga kamu tidak perlu login ulang setiap kali menjalankan notebook.


In [ ]:
connection = openeo.connect(BACKEND_URL)
connection = connection.authenticate_oidc()

print("Berhasil terhubung dan terautentikasi ke:", BACKEND_URL)


## 4. Fungsi Ekstraksi untuk Satu Polutan

Sebelum itu, fungsi kecil `safe_download()` disiapkan untuk menghindari
`PermissionError` di Windows. Masalah ini terjadi jika file `.nc` dari
percobaan sebelumnya masih "terkunci" karena sempat dibuka dengan
`xr.open_dataset()` tanpa ditutup. `safe_download()` mencoba menghapus file
lama terlebih dahulu sebelum menulis yang baru, dan memberi pesan yang jelas
jika file tersebut masih terkunci oleh proses lain.

Fungsi `extract_pollutant()` melakukan seluruh proses openEO untuk satu
polutan:

1. `load_collection()`, memuat koleksi `SENTINEL_5P_L2`, dibatasi oleh
   bounding box kasar, rentang waktu, dan satu band polutan.
2. `aggregate_temporal_period(period="day", reducer="mean")`, agregasi
   temporal harian (mean), sehingga beberapa lintasan satelit dalam satu hari
   dirata-ratakan menjadi satu nilai per hari.
3. `aggregate_spatial(geometries=..., reducer="mean")`, agregasi spasial,
   merata-ratakan seluruh piksel di dalam polygon asli Bangkalan menjadi satu
   nilai per hari (menghasilkan vector cube, yaitu deret waktu).
4. `save_result(format="netCDF")`, menetapkan format keluaran job sebagai
   NetCDF.
5. Job dijalankan sebagai batch job (`create_job` dan `start_and_wait`),
   karena ekstraksi setahun data deret waktu biasanya memakan waktu lebih
   lama daripada proses sinkron.
6. Hasil job diunduh ke folder `NC_DIR` dengan nama file
   `bangkalan_<nama_polutan>.nc`.


In [ ]:
def safe_download(job, target_path):
    """Mengunduh hasil job, menghindari PermissionError akibat file lama
    yang masih terkunci (umum terjadi di Windows).
    """
    if os.path.exists(target_path):
        try:
            os.remove(target_path)
        except PermissionError:
            print(
                f"PERINGATAN: {target_path} masih terkunci oleh proses lain "
                "(misalnya dataset yang belum ditutup dengan .close(), atau "
                "sedang dibuka aplikasi lain). Coba restart kernel notebook "
                "lalu jalankan ulang sel ini."
            )
            raise

    job.get_results().download_file(target=target_path)
    return target_path


In [ ]:
def extract_pollutant(connection, pollutant_name, band_name,
                       bbox, polygon, start_date, end_date, output_dir):
    """Menarik satu polutan Sentinel-5P L2 untuk area & rentang waktu tertentu.

    Parameters
    ----------
    connection : openeo.Connection
        Koneksi openEO yang sudah terautentikasi.
    pollutant_name : str
        Nama pendek polutan, dipakai untuk penamaan file (misal "NO2").
    band_name : str
        Nama band pada koleksi SENTINEL_5P_L2 (misal "NO2").
    bbox : dict
        Bounding box kasar dengan key west/east/south/north.
    polygon : dict
        Geometry GeoJSON (Polygon) asli Bangkalan untuk agregasi spasial.
    start_date, end_date : str
        Rentang waktu dalam format "YYYY-MM-DD".
    output_dir : str
        Folder tujuan penyimpanan file NetCDF.

    Returns
    -------
    str
        Path file NetCDF hasil ekstraksi.
    """
    print(f"[{pollutant_name}] Membuat data cube ...")
    cube = connection.load_collection(
        "SENTINEL_5P_L2",
        spatial_extent=bbox,
        temporal_extent=[start_date, end_date],
        bands=[band_name],
    )

    print(f"[{pollutant_name}] Agregasi temporal harian (mean) ...")
    cube = cube.aggregate_temporal_period(period="day", reducer="mean")

    print(f"[{pollutant_name}] Agregasi spasial berdasarkan polygon asli Bangkalan (mean) ...")
    cube = cube.aggregate_spatial(geometries=polygon, reducer="mean")

    result = cube.save_result(format="netCDF")

    print(f"[{pollutant_name}] Mengirim batch job ke backend openEO ...")
    job = result.create_job(title=f"bangkalan_{pollutant_name}")
    job.start_and_wait()

    output_path = os.path.join(output_dir, f"bangkalan_{pollutant_name}.nc")
    safe_download(job, output_path)
    print(f"[{pollutant_name}] Selesai -> {output_path}")

    return output_path


## 5. Uji Coba Tunggal dan Diagnostik Nilai 0

Sebelum menjalankan ekstraksi penuh setahun untuk keempat polutan, sangat
disarankan menjalankan **satu uji coba kecil** terlebih dahulu, misalnya
untuk NO2 pada rentang waktu singkat (satu minggu). Hasil hasil yang seluruhnya
bernilai 0 pada tahap sebelumnya umumnya disebabkan oleh salah satu dari hal
berikut.

1. **Nama band tidak sesuai.** Nama band pada koleksi `SENTINEL_5P_L2`
   bersifat case sensitive. Gunakan `connection.describe_collection(...)`
   untuk memastikan nama band yang benar sebelum melakukan ekstraksi penuh.
2. **Nilai kosong (NaN) yang tidak disadari berubah menjadi 0.** Data
   Sentinel-5P sering memiliki hari tanpa data valid akibat tutupan awan.
   Jika nilai NaN tersebut tidak sengaja diisi dengan `fillna(0)` pada suatu
   tahap pemrosesan, hasil akhirnya akan terlihat seolah olah bernilai 0,
   padahal sebenarnya data tidak ada, bukan nol.
3. **Nilai konsentrasi memang sangat kecil.** Satuan konsentrasi trace gas
   pada Sentinel-5P berada pada orde 1e-4 hingga 1e-6. Jika dicetak dengan
   pembulatan yang terlalu sedikit angka desimal, nilai seperti itu bisa
   terlihat seperti 0.000000 padahal bukan nol sesungguhnya. Cetak nilai
   dengan notasi ilmiah untuk memastikan.
4. **Cakupan area tidak tepat sasaran.** Jika bounding box atau polygon yang
   dipakai meleset dari wilayah yang dimaksud, misalnya sebagian besar jatuh
   di area laut yang tidak memiliki lintasan pengukuran relevan, hasil
   agregasi bisa didominasi oleh piksel kosong.
5. **Ada error atau warning pada job yang tidak terlihat.** Job batch bisa
   saja selesai tanpa error fatal namun tetap menghasilkan data kosong.
   Periksa `job.logs()` untuk melihat pesan yang mungkin terlewat.
6. **Penyaringan kualitas (QA) internal pada backend.** Backend openEO
   Copernicus Data Space saat ini menyaring piksel dengan `qa_value` di
   bawah 0.5 menjadi NaN sebelum data sampai ke pengguna (dikonfirmasi pada
   forum resmi Copernicus Data Space, topik "Quality factor for
   Sentinel-5P L2 SO2 products"). Untuk wilayah kecil dan berawan, ini bisa
   membuat sebagian atau seluruh piksel pada suatu hari tersaring habis.

**Sel pertama** di bawah menguji data paling mentah, yaitu grid piksel NO2
tanpa agregasi spasial maupun temporal, hanya untuk satu hari. Ini
mengisolasi apakah masalah sudah muncul sejak `load_collection`, sebelum
proses agregasi apa pun dilakukan.


In [ ]:
# Uji coba paling mendasar: unduh grid piksel mentah NO2 untuk satu hari saja,
# TANPA aggregate_temporal_period maupun aggregate_spatial. Ini mengisolasi
# apakah data pada sumbernya memang sudah nol/tidak ada, atau justru berubah
# menjadi 0 pada tahap agregasi selanjutnya.
raw_cube = connection.load_collection(
    "SENTINEL_5P_L2",
    spatial_extent=bangkalan_bbox,
    temporal_extent=["2025-08-24", "2025-08-25"],
    bands=[POLLUTANTS["NO2"]],
)
raw_job = raw_cube.save_result(format="netCDF").create_job(title="raw_grid_test_NO2")
raw_job.start_and_wait()

raw_test_path = os.path.join(NC_DIR, "raw_grid_test_NO2.nc")
safe_download(raw_job, raw_test_path)

with xr.open_dataset(raw_test_path) as ds_raw:
    print(ds_raw)

    # Pilih hanya variabel data numerik multi-dimensi (grid piksel), bukan
    # variabel bantu seperti "crs" (metadata sistem koordinat) yang ikut
    # tersimpan di NetCDF tapi bukan data polutan.
    numeric_vars = [
        v for v in ds_raw.data_vars
        if np.issubdtype(ds_raw[v].dtype, np.number) and ds_raw[v].ndim >= 2
    ]
    if not numeric_vars:
        raise ValueError(
            f"Tidak ditemukan variabel data numerik pada NetCDF. "
            f"Variabel yang tersedia: {list(ds_raw.data_vars)}"
        )
    var_name = numeric_vars[0]
    print(f"\nVariabel data yang dipakai: '{var_name}' (variabel lain diabaikan: "
          f"{[v for v in ds_raw.data_vars if v != var_name]})")

    raw_values = ds_raw[var_name].values
    print()
    print("Jumlah piksel total   :", raw_values.size)
    print("Jumlah piksel NaN     :", int(pd.isna(raw_values).sum()))
    print("Jumlah piksel 0.0     :", int((raw_values == 0).sum()))
    print("Jumlah piksel valid   :", int((~pd.isna(raw_values) & (raw_values != 0)).sum()))
    print("Nilai minimum (valid) : {:.3e}".format(
        pd.Series(raw_values.flatten()).replace(0, pd.NA).min(skipna=True)
    ))
    print("Nilai maksimum (valid): {:.3e}".format(
        pd.Series(raw_values.flatten()).replace(0, pd.NA).max(skipna=True)
    ))

# Dataset ditutup otomatis oleh blok "with" di atas, sehingga file .nc tidak
# akan terkunci saat sel ini dijalankan ulang (menghindari PermissionError
# yang umum terjadi di Windows).
#
# Jika "Jumlah piksel valid" di atas menunjukkan 0 (tidak ada satu pun piksel
# bernilai selain NaN atau 0.0) untuk grid mentah satu hari ini, kemungkinan
# besar masalah ada pada request itu sendiri (bbox/polygon, nama band, atau
# koleksi), bukan pada langkah agregasi. Jika grid mentah ini justru
# menunjukkan banyak piksel valid dan bervariasi, artinya masalah baru muncul
# pada langkah aggregate_temporal_period atau aggregate_spatial berikutnya.


**Sel kedua** di bawah menguji pipeline lengkap (agregasi temporal dan
spasial) untuk NO2, tetapi hanya satu minggu pertama sebagai uji coba,
sebelum menjalankan ekstraksi penuh setahun untuk keempat polutan.


In [ ]:
# Uji coba: ekstraksi NO2 untuk satu minggu pertama saja
test_nc_path = extract_pollutant(
    connection=connection,
    pollutant_name="NO2_TEST",
    band_name=POLLUTANTS["NO2"],
    bbox=bangkalan_bbox,
    polygon=bangkalan_polygon,
    start_date=START_DATE,
    end_date="2025-08-31",
    output_dir=NC_DIR,
)

# Diagnostik: periksa nilai mentah sebelum dipakai lebih lanjut
with xr.open_dataset(test_nc_path) as ds_test:
    print(ds_test)

    # Hanya proses variabel data numerik (abaikan variabel bantu seperti "crs")
    value_vars = [
        v for v in ds_test.data_vars
        if np.issubdtype(ds_test[v].dtype, np.number)
    ]
    print()
    print("Variabel data numerik yang diproses:", value_vars)

    for var in value_vars:
        values = ds_test[var].values
        print(f"\nVariabel '{var}':")
        print("  jumlah data       :", values.size)
        print("  jumlah NaN        :", int(pd.isna(values).sum()))
        print("  jumlah bernilai 0 :", int((values == 0).sum()))
        print("  nilai minimum     : {:.3e}".format(pd.Series(values.flatten()).min(skipna=True)))
        print("  nilai maksimum    : {:.3e}".format(pd.Series(values.flatten()).max(skipna=True)))
        print("  nilai rata rata   : {:.3e}".format(pd.Series(values.flatten()).mean(skipna=True)))

# Dataset ditutup otomatis oleh blok "with" di atas (menghindari
# PermissionError di Windows saat file ini ditulis ulang nanti).
#
# Jika seluruh nilai memang 0 (bukan NaN dan bukan angka sangat kecil),
# periksa kembali nama band lewat baris berikut sebelum melanjutkan.
# print(connection.describe_collection("SENTINEL_5P_L2"))


## 6. Menjalankan Ekstraksi untuk Semua Polutan (Dipecah per 3 Bulan)

**Update diagnosis:** setelah ditelusuri lebih lanjut lewat `job.logs()`,
akar masalah "hasil 0 semua" ternyata bukan (hanya) soal panjang rentang
waktu, melainkan **rate limit (HTTP 429 Too Many Requests)** dari layanan
tile Sentinel Hub yang dipakai backend openEO Copernicus Data Space. Saat
banyak job dikirim berurutan dengan jeda singkat, sebagian permintaan tile
untuk tanggal tertentu gagal diambil setelah beberapa kali percobaan ulang
("Not attempting to retry unrecoverable error"), dan tanggal tersebut
berakhir tanpa data pada hasil akhir, meskipun status job tetap
"finished" sukses.

**Solusinya ada dua lapis:**
1. Rentang setahun tetap dipecah menjadi beberapa bagian 3 bulanan (supaya
   satu job tidak perlu mengambil terlalu banyak tile sekaligus).
2. Jeda antar job diperpanjang jauh lebih lama (30 detik, bukan 2 detik),
   dan setiap hasil bagian diperiksa otomatis: jika proporsi nilai 0 pada
   suatu bagian mencurigakan tinggi (lebih dari 30%, indikasi kena rate
   limit), bagian tersebut **otomatis dicoba ulang** (maksimal 2 kali
   percobaan tambahan) sebelum akhirnya diterima apa adanya.


In [ ]:
def make_date_chunks(start_date, end_date, chunk_months=3):
    """Membagi rentang tanggal menjadi beberapa bagian yang lebih pendek,
    supaya satu job tidak perlu mengambil terlalu banyak tile sekaligus dari
    layanan Sentinel Hub (mengurangi risiko kena rate limit / HTTP 429).
    """
    chunks = []
    current = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    while current < end:
        chunk_end = min(current + pd.DateOffset(months=chunk_months), end)
        chunks.append((current.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")))
        current = chunk_end
    return chunks


def fraction_zero(nc_path):
    """Menghitung proporsi nilai persis 0.0 pada satu file NetCDF hasil
    aggregate_spatial, dipakai sebagai indikator kecurigaan rate limit.
    """
    with xr.open_dataset(nc_path) as ds:
        numeric_vars = [
            v for v in ds.data_vars
            if np.issubdtype(ds[v].dtype, np.number)
        ]
        if not numeric_vars:
            return 1.0
        values = ds[numeric_vars[0]].values
        non_nan = values[~pd.isna(values)]
        if len(non_nan) == 0:
            return 1.0
        return float((non_nan == 0).sum()) / len(non_nan)


DATE_CHUNKS = make_date_chunks(START_DATE, END_DATE, chunk_months=3)

print(f"Rentang {START_DATE} sampai {END_DATE} dibagi menjadi {len(DATE_CHUNKS)} bagian:")
for i, (chunk_start, chunk_end) in enumerate(DATE_CHUNKS, start=1):
    print(f"  Bagian {i}: {chunk_start} sampai {chunk_end}")


In [ ]:
DELAY_BETWEEN_JOBS = 30       # detik, jeda antar job agar tidak kena rate limit
MAX_RETRY_PER_CHUNK = 2       # percobaan tambahan jika hasil mencurigakan
ZERO_FRACTION_THRESHOLD = 0.3  # di atas ini, bagian dianggap mencurigakan

nc_paths = {}  # {nama_polutan: [daftar path file .nc per bagian]}

for pollutant_name, band_name in POLLUTANTS.items():
    nc_paths[pollutant_name] = []

    for i, (chunk_start, chunk_end) in enumerate(DATE_CHUNKS, start=1):
        chunk_label = f"{pollutant_name}_bagian{i}"
        attempt = 0
        accepted_path = None

        while attempt <= MAX_RETRY_PER_CHUNK:
            attempt += 1
            try:
                nc_path = extract_pollutant(
                    connection=connection,
                    pollutant_name=chunk_label,
                    band_name=band_name,
                    bbox=bangkalan_bbox,
                    polygon=bangkalan_polygon,
                    start_date=chunk_start,
                    end_date=chunk_end,
                    output_dir=NC_DIR,
                )
                zero_frac = fraction_zero(nc_path)
                print(f"[{chunk_label}] Percobaan {attempt}: proporsi nilai 0.0 = {zero_frac:.1%}")

                if zero_frac <= ZERO_FRACTION_THRESHOLD:
                    accepted_path = nc_path
                    break
                elif attempt <= MAX_RETRY_PER_CHUNK:
                    print(f"[{chunk_label}] Proporsi 0.0 terlalu tinggi (indikasi rate limit), "
                          f"menunggu lalu mencoba ulang ...")
                    time.sleep(DELAY_BETWEEN_JOBS * 2)
                else:
                    print(f"[{chunk_label}] Tetap dipakai walau proporsi 0.0 tinggi "
                          f"(sudah {MAX_RETRY_PER_CHUNK} kali percobaan ulang).")
                    accepted_path = nc_path
            except Exception as exc:
                print(f"[{chunk_label}] GAGAL diekstraksi (percobaan {attempt}): {exc}")
                if attempt <= MAX_RETRY_PER_CHUNK:
                    time.sleep(DELAY_BETWEEN_JOBS * 2)

        if accepted_path:
            nc_paths[pollutant_name].append(accepted_path)

        # Jeda panjang antar job agar tidak membanjiri layanan tile Sentinel Hub
        time.sleep(DELAY_BETWEEN_JOBS)

print()
print("Ringkasan file NetCDF yang berhasil dibuat per polutan:")
for pollutant_name, paths in nc_paths.items():
    print(f"  - {pollutant_name}: {len(paths)} dari {len(DATE_CHUNKS)} bagian berhasil")


## 7. Menggabungkan Semua Bagian dan Polutan Menjadi Satu Tabel

Alih alih menyimpan satu file CSV terpisah per polutan (atau per bagian),
seluruh hasil digabungkan menjadi **satu tabel tunggal**, mirip struktur
tabel pada basis data, dengan satu baris per tanggal dan satu kolom per
polutan.

Fungsi `nc_to_series()` membuka satu file NetCDF dan mengembalikannya
sebagai `pandas.Series` bertipe waktu. Fungsi `nc_chunks_to_series()`
menggabungkan beberapa bagian (chunk) milik satu polutan yang sama menjadi
satu deret waktu utuh. Seluruh series per polutan kemudian digabungkan
dengan `pd.concat(axis=1)` berdasarkan tanggal yang sama, menghasilkan satu
`DataFrame` dengan kolom `date`, `NO2`, `CO`, `SO2`, dan `CH4`.

> **Koreksi diagnosis sebelumnya.** Setelah ditelusuri lebih lanjut,
> sebagian besar nilai 0.0 yang muncul pada versi awal notebook ini
> ternyata **bukan** disebabkan oleh rate limit ataupun penyaringan
> kualitas (QA) backend, melainkan bug pada fungsi `nc_to_series()`
> versi sebelumnya: NetCDF hasil `aggregate_spatial` memuat beberapa kolom
> numerik sekaligus (`feature`, `NO2`/`CO`/`SO2`/`CH4`, `lat`, `lon`), dan
> kode lama secara tidak sengaja mengambil kolom `feature` (indeks fitur,
> selalu bernilai 0) alih alih kolom polutan yang sebenarnya. Versi
> `nc_to_series()` di bawah ini sudah diperbaiki dengan mencocokkan nama
> band secara eksplisit, sehingga kolom yang benar selalu terpilih.
>
> Meskipun demikian, nilai 0.0 yang genuinely berasal dari penyaringan QA
> backend tetap mungkin terjadi sesekali (dikonfirmasi pada forum resmi
> Copernicus Data Space, topik "Quality factor for Sentinel-5P L2 SO2
> products"), sehingga nilai 0.0 yang tersisa setelah perbaikan ini tetap
> diperlakukan sebagai data hilang, bukan pengukuran valid, dan dikonversi
> menjadi NaN sebelum disimpan.


In [ ]:
def nc_to_series(nc_path, pollutant_name, band_name):
    """Membuka satu NetCDF hasil aggregate_spatial dan mengembalikannya sebagai Series.

    Kolom nilai dipilih dengan mencocokkan nama band secara eksplisit
    (misal "NO2"), bukan menebak kolom numerik pertama yang ditemukan.
    Ini penting karena NetCDF hasil aggregate_spatial juga memuat kolom
    lain seperti 'feature' (indeks fitur, selalu 0), 'lat', dan 'lon' yang
    ikut lolos filter "kolom numerik" dan bisa salah terpilih jika hanya
    mengambil kandidat pertama.
    """
    with xr.open_dataset(nc_path) as ds:
        df = ds.to_dataframe().reset_index()

    time_col_candidates = [c for c in df.columns if "time" in c.lower() or c.lower() == "t"]
    time_col = time_col_candidates[0] if time_col_candidates else df.columns[0]

    if band_name in df.columns:
        value_col = band_name
    else:
        exclude = {"feature", "lat", "lon", "latitude", "longitude", "geometry"}
        value_col_candidates = [
            c for c in df.columns
            if c != time_col and c.lower() not in exclude
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        value_col = value_col_candidates[0] if value_col_candidates else df.columns[-1]

    series = df.set_index(pd.to_datetime(df[time_col]))[value_col]
    series.name = pollutant_name
    series.index.name = "date"
    return series.sort_index()


def nc_chunks_to_series(nc_path_list, pollutant_name, band_name):
    """Menggabungkan beberapa file NetCDF (per bagian tanggal) milik satu
    polutan menjadi satu deret waktu utuh, tanpa tanggal duplikat.
    """
    parts = [nc_to_series(path, pollutant_name, band_name) for path in nc_path_list]
    combined = pd.concat(parts)
    combined = combined[~combined.index.duplicated(keep="first")]
    return combined.sort_index()


series_list = []
for name, paths in nc_paths.items():
    if not paths:
        print(f"[{name}] DILEWATI: tidak ada file NetCDF yang berhasil diunduh "
              f"untuk polutan ini (0 dari {len(DATE_CHUNKS)} bagian berhasil). "
              f"Cek pesan 'GAGAL diekstraksi' di output Bagian 6 di atas.")
        continue
    series_list.append(nc_chunks_to_series(paths, name, POLLUTANTS[name]))

if not series_list:
    raise RuntimeError(
        "Tidak ada satupun polutan yang berhasil diekstraksi sama sekali. "
        "Periksa log error pada Bagian 6 sebelum melanjutkan."
    )

combined_df = pd.concat(series_list, axis=1).reset_index()
combined_df = combined_df.sort_values("date").reset_index(drop=True)

pollutant_cols = [name for name, paths in nc_paths.items() if paths]

# Nilai persis 0.0 diperlakukan sebagai data hilang (lihat catatan di atas),
# karena secara fisis konsentrasi trace gas atmosfer tidak pernah benar
# benar nol; nilai 0.0 yang muncul kemungkinan besar adalah artefak dari
# penyaringan kualitas (QA) pada backend, bukan pengukuran yang valid.
n_zero_before = (combined_df[pollutant_cols] == 0).sum()
combined_df[pollutant_cols] = combined_df[pollutant_cols].where(combined_df[pollutant_cols] != 0)

print("Jumlah nilai 0.0 yang dikonversi menjadi data hilang (NaN):")
print(n_zero_before)

combined_df.to_csv(COMBINED_CSV_PATH, index=False)

print()
print(f"Tabel gabungan disimpan -> {COMBINED_CSV_PATH}")
print(f"Jumlah baris : {len(combined_df)}")
print("Jumlah data valid per kolom:")
print(combined_df[pollutant_cols].notna().sum())
combined_df.head()


NameError: name 'bash' is not defined

In [6]:
import os
import pandas as pd
import xarray as xr

# --- Definisi ulang konfigurasi (harus sama persis dengan saat ekstraksi) ---
NC_DIR = "../data/nc/"
COMBINED_CSV_PATH = "data_polutan_bangkalan.csv"
POLLUTANTS = {"NO2": "NO2", "CO": "CO", "SO2": "SO2", "CH4": "CH4"}

START_DATE = "2025-08-24"
END_DATE = "2026-08-24"

def make_date_chunks(start_date, end_date, chunk_months=3):
    chunks = []
    current = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    while current < end:
        chunk_end = min(current + pd.DateOffset(months=chunk_months), end)
        chunks.append((current.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")))
        current = chunk_end
    return chunks

DATE_CHUNKS = make_date_chunks(START_DATE, END_DATE, chunk_months=3)

# --- Bangun ulang nc_paths dari file yang sudah ada di disk ---
nc_paths = {}
for pollutant_name in POLLUTANTS.keys():
    paths = [
        os.path.join(NC_DIR, f"bangkalan_{pollutant_name}_bagian{i}.nc")
        for i in range(1, len(DATE_CHUNKS) + 1)
    ]
    nc_paths[pollutant_name] = [p for p in paths if os.path.exists(p)]
    print(f"{pollutant_name}: {len(nc_paths[pollutant_name])} dari {len(DATE_CHUNKS)} file ditemukan")

# --- Fungsi baca & gabung yang sudah diperbaiki ---
def nc_to_series(nc_path, pollutant_name, band_name):
    with xr.open_dataset(nc_path) as ds:
        df = ds.to_dataframe().reset_index()

    time_col_candidates = [c for c in df.columns if "time" in c.lower() or c.lower() == "t"]
    time_col = time_col_candidates[0] if time_col_candidates else df.columns[0]

    if band_name in df.columns:
        value_col = band_name
    else:
        exclude = {"feature", "lat", "lon", "latitude", "longitude", "geometry"}
        value_col_candidates = [
            c for c in df.columns
            if c != time_col and c.lower() not in exclude
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        value_col = value_col_candidates[0] if value_col_candidates else df.columns[-1]

    series = df.set_index(pd.to_datetime(df[time_col]))[value_col]
    series.name = pollutant_name
    series.index.name = "date"
    return series.sort_index()

def nc_chunks_to_series(nc_path_list, pollutant_name, band_name):
    parts = [nc_to_series(path, pollutant_name, band_name) for path in nc_path_list]
    combined = pd.concat(parts)
    combined = combined[~combined.index.duplicated(keep="first")]
    return combined.sort_index()

# --- Gabungkan ---
series_list = []
for name, paths in nc_paths.items():
    if not paths:
        print(f"[{name}] DILEWATI: tidak ada file NetCDF.")
        continue
    series_list.append(nc_chunks_to_series(paths, name, POLLUTANTS[name]))

combined_df = pd.concat(series_list, axis=1).reset_index()
combined_df = combined_df.sort_values("date").reset_index(drop=True)

pollutant_cols = [name for name in nc_paths.keys() if nc_paths[name]]

n_zero_before = (combined_df[pollutant_cols] == 0).sum()
combined_df[pollutant_cols] = combined_df[pollutant_cols].where(combined_df[pollutant_cols] != 0)

print("\nJumlah nilai 0.0 yang dikonversi menjadi data hilang (NaN):")
print(n_zero_before)

combined_df.to_csv(COMBINED_CSV_PATH, index=False)

print(f"\nTabel gabungan disimpan -> {COMBINED_CSV_PATH}")
print(f"Jumlah baris : {len(combined_df)}")
print("Jumlah data valid per kolom:")
print(combined_df[pollutant_cols].notna().sum())
combined_df.head(10)

import os
import pandas as pd
import xarray as xr

# --- Definisi ulang konfigurasi (harus sama persis dengan saat ekstraksi) ---
NC_DIR = "../data/nc/"
COMBINED_CSV_PATH = "data_polutan_bangkalan.csv"
POLLUTANTS = {"NO2": "NO2", "CO": "CO", "SO2": "SO2", "CH4": "CH4"}

START_DATE = "2025-08-24"
END_DATE = "2026-08-24"

def make_date_chunks(start_date, end_date, chunk_months=3):
    chunks = []
    current = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)
    while current < end:
        chunk_end = min(current + pd.DateOffset(months=chunk_months), end)
        chunks.append((current.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")))
        current = chunk_end
    return chunks

DATE_CHUNKS = make_date_chunks(START_DATE, END_DATE, chunk_months=3)

# --- Bangun ulang nc_paths dari file yang sudah ada di disk ---
nc_paths = {}
for pollutant_name in POLLUTANTS.keys():
    paths = [
        os.path.join(NC_DIR, f"bangkalan_{pollutant_name}_bagian{i}.nc")
        for i in range(1, len(DATE_CHUNKS) + 1)
    ]
    nc_paths[pollutant_name] = [p for p in paths if os.path.exists(p)]
    print(f"{pollutant_name}: {len(nc_paths[pollutant_name])} dari {len(DATE_CHUNKS)} file ditemukan")

# --- Fungsi baca & gabung yang sudah diperbaiki ---
def nc_to_series(nc_path, pollutant_name, band_name):
    with xr.open_dataset(nc_path) as ds:
        df = ds.to_dataframe().reset_index()

    time_col_candidates = [c for c in df.columns if "time" in c.lower() or c.lower() == "t"]
    time_col = time_col_candidates[0] if time_col_candidates else df.columns[0]

    if band_name in df.columns:
        value_col = band_name
    else:
        exclude = {"feature", "lat", "lon", "latitude", "longitude", "geometry"}
        value_col_candidates = [
            c for c in df.columns
            if c != time_col and c.lower() not in exclude
            and pd.api.types.is_numeric_dtype(df[c])
        ]
        value_col = value_col_candidates[0] if value_col_candidates else df.columns[-1]

    series = df.set_index(pd.to_datetime(df[time_col]))[value_col]
    series.name = pollutant_name
    series.index.name = "date"
    return series.sort_index()

def nc_chunks_to_series(nc_path_list, pollutant_name, band_name):
    parts = [nc_to_series(path, pollutant_name, band_name) for path in nc_path_list]
    combined = pd.concat(parts)
    combined = combined[~combined.index.duplicated(keep="first")]
    return combined.sort_index()

# --- Gabungkan ---
series_list = []
for name, paths in nc_paths.items():
    if not paths:
        print(f"[{name}] DILEWATI: tidak ada file NetCDF.")
        continue
    series_list.append(nc_chunks_to_series(paths, name, POLLUTANTS[name]))

combined_df = pd.concat(series_list, axis=1).reset_index()
combined_df = combined_df.sort_values("date").reset_index(drop=True)

pollutant_cols = [name for name in nc_paths.keys() if nc_paths[name]]

n_zero_before = (combined_df[pollutant_cols] == 0).sum()
combined_df[pollutant_cols] = combined_df[pollutant_cols].where(combined_df[pollutant_cols] != 0)

print("\nJumlah nilai 0.0 yang dikonversi menjadi data hilang (NaN):")
print(n_zero_before)

combined_df.to_csv(COMBINED_CSV_PATH, index=False)

print(f"\nTabel gabungan disimpan -> {COMBINED_CSV_PATH}")
print(f"Jumlah baris : {len(combined_df)}")
print("Jumlah data valid per kolom:")
print(combined_df[pollutant_cols].notna().sum())
combined_df.head(10)

NO2: 4 dari 4 file ditemukan
CO: 4 dari 4 file ditemukan
SO2: 4 dari 4 file ditemukan
CH4: 4 dari 4 file ditemukan

Jumlah nilai 0.0 yang dikonversi menjadi data hilang (NaN):
NO2    0
CO     0
SO2    0
CH4    0
dtype: int64

Tabel gabungan disimpan -> data_polutan_bangkalan.csv
Jumlah baris : 314
Jumlah data valid per kolom:
NO2    278
CO     270
SO2    298
CH4     71
dtype: int64
NO2: 4 dari 4 file ditemukan
CO: 4 dari 4 file ditemukan
SO2: 4 dari 4 file ditemukan
CH4: 4 dari 4 file ditemukan

Jumlah nilai 0.0 yang dikonversi menjadi data hilang (NaN):
NO2    0
CO     0
SO2    0
CH4    0
dtype: int64

Tabel gabungan disimpan -> data_polutan_bangkalan.csv
Jumlah baris : 314
Jumlah data valid per kolom:
NO2    278
CO     270
SO2    298
CH4     71
dtype: int64


,date,NO2,CO,SO2,CH4
0,2025-08-24,0.000021,0.026651,0.000177,1850.206787
1,2025-08-25,0.000013,0.027321,0.000241,1878.566479
2,2025-08-26,0.000039,NaN,-0.000008,NaN
3,2025-08-27,0.000032,0.029484,-0.000066,1885.140747
4,2025-08-28,0.000018,0.024329,0.000027,NaN
5,2025-08-29,0.000024,0.025603,0.000112,1846.600549
6,2025-08-30,0.000016,0.025166,-0.000008,NaN
7,2025-08-31,0.000008,0.025972,-0.000047,NaN
8,2025-09-01,0.000014,0.027356,0.000045,NaN
9,2025-09-02,0.000010,0.024187,-0.000041,NaN


## Selesai

Setelah notebook ini dijalankan, kamu akan memiliki:

- `../data/nc/bangkalan_<POLUTAN>.nc`, data mentah hasil openEO per polutan.
- `data_polutan_bangkalan.csv`, **satu tabel gabungan** (satu folder dengan
  notebook ini) berisi kolom `date`, `NO2`, `CO`, `SO2`, dan `CH4`, siap
  dipakai langsung pada notebook analisis.

Struktur tabel gabungan kurang lebih sebagai berikut.

| date | NO2 | CO | SO2 | CH4 |
|------|-----|----|----|-----|
| 2025-08-24 | 0.000123 | 0.021 | 0.0004 | 0.00019 |
| 2025-08-25 | 0.000119 | 0.020 | 0.0003 | 0.00018 |
| ... | ... | ... | ... | ... |

Lanjutkan ke notebook **`5-analisis-bangkalan.ipynb`** untuk melakukan
eksplorasi, visualisasi peta, penanganan missing values, dan deteksi outlier
menggunakan Isolation Forest.
